In [1]:
!pip install xgboost gradio


In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBRegressor
import numpy as np

# Upload dataset manually
from google.colab import files
uploaded = files.upload()

df = pd.read_csv(list(uploaded.keys())[0])

# Handle missing values
df.fillna(df.mean(numeric_only=True), inplace=True)

# Encode categorical columns
le_gender = LabelEncoder()
le_edu = LabelEncoder()
le_job = LabelEncoder()

df['Gender'] = le_gender.fit_transform(df['Gender'])
df['Education Level'] = le_edu.fit_transform(df['Education Level'])
df['Job Title'] = le_job.fit_transform(df['Job Title'])

X = df[['Age','Gender','Education Level','Job Title','Years of Experience']]
y = df['Salary']

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

model = XGBRegressor()
model.fit(X_train,y_train)

print("Model Trained Successfully ✅")

Saving Employee_Salary_with_Skills.csv to Employee_Salary_with_Skills.csv
Model Trained Successfully ✅


In [3]:
from sklearn.metrics import r2_score, mean_absolute_error

y_pred = model.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print("Model Performance:")
print("R2 Score:", round(r2, 3))
print("MAE:", round(mae, 2))


Model Performance:
R2 Score: 0.917
MAE: 9182.92


In [4]:
from sklearn.metrics import mean_squared_error
import numpy as np

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(rmse)

14147.025597742471


In [5]:
import gradio as gr
import pandas as pd
import matplotlib.pyplot as plt

# ================= CSS =================
custom_css = """
body {
    background: #f1f5f9;
    font-family: 'Segoe UI', sans-serif;
}

.gradio-container {
    background: transparent !important;
}

div[data-testid="column"] {
    background: white;
    border-radius: 14px;
    padding: 20px;
    margin-top: 12px;
    border: 1px solid #e2e8f0;
    box-shadow: 0 4px 12px rgba(0,0,0,0.05);
    position: relative;
    z-index: 1;
}

button {
    border-radius: 8px !important;
    background: #2563eb !important;
    color: white !important;
    font-weight: 600;
    border: none !important;
    transition: all 0.25s ease;
}

button:hover {
    background: #1d4ed8 !important;
    transform: translateY(-2px);
}

h1 {
    color: #0f172a !important;
    text-align: center;
    font-weight: 800;
}

h2, h3 {
    color: #1e293b !important;
    font-weight: 700;
}

p, label {
    color: #334155 !important;
}

/* FIX INPUT (exclude checkbox) */
input:not([type="checkbox"]), select {
    background: #f8fafc !important;
    color: #0f172a !important;
    border: 1px solid #cbd5f5 !important;
    border-radius: 8px !important;
}

/* FIX CHECKBOX CLICK */
input[type="checkbox"] {
    pointer-events: auto !important;
    cursor: pointer !important;
    accent-color: #2563eb;
}
"""

# ================= DATA =================
df = pd.read_csv("Employee_Salary_with_Skills.csv")
df.columns = df.columns.str.strip()

job_roles = sorted(df["Job Title"].dropna().astype(str).unique().tolist())

# ================= SKILLS =================
if "Skills" not in df.columns:
    df["Skills"] = "Python, Communication"

skills_list = sorted(list(set(
    s.strip()
    for row in df["Skills"].dropna()
    for s in str(row).split(",")
    if s.strip() != ""
)))

# ================= USERS =================
users = {"admin": "1234"}

# ================= PAGE CONTROL =================
def show(p):
    return [gr.update(visible=(i == p)) for i in range(12)]

# ================= AUTH =================
def login(u, p):
    if u in users and users[u] == p:
        return "✅ Login Successful", *show(5)
    return "❌ Invalid Credentials", *show(2)

def signup(username, password, confirm):
    if username in users:
        return "❌ User already exists"
    if password != confirm:
        return "❌ Passwords do not match"
    users[username] = password
    return "✅ Registered Successfully!"

def forgot_password(username):
    if username in users:
        return f"🔑 Password: {users[username]}"
    return "❌ User not found"

def toggle_password(show):
    return gr.update(type="text" if show else "password")

# ================= PREDICTION =================
def predict(age, gender, edu, job, exp, skills):

    salary = int(30000 + exp * 7000 + len(skills)*2000)

    plt.figure()
    plt.scatter(df["Years of Experience"], df["Salary"])
    plt.xlabel("Experience")
    plt.ylabel("Salary")
    plt.title("Experience vs Salary")
    chart = plt.gcf()

    avg_salary = int(df["Salary"].mean())
    role_avg = int(df[df["Job Title"] == job]["Salary"].mean())

    explanation = f"""
# 📊 AI Salary Justification Report

## 👤 Profile Summary
- Age: {age}
- Gender: {gender}
- Education: {edu}
- Job Role: {job}
- Experience: {exp} years
- Skills: {", ".join(skills) if skills else "None"}

## 💰 Predicted Salary
### 👉 ₹{salary}

## 📈 Detailed Justification

### 1. Experience-Based Growth
Salary increases consistently with experience in the dataset.
You fall under:
👉 **{'Entry Level (0–3 yrs)' if exp<3 else 'Mid Level (3–7 yrs)' if exp<7 else 'Senior Level (7+ yrs)'}**

### 2. Role-Based Comparison
- Average salary for **{job}**: ₹{role_avg}

👉 Your salary is:
{'✔ Higher than role average (strong position)' if salary > role_avg else '⚠ Slightly below role average'}

### 3. Market Benchmark
- Overall average salary: ₹{avg_salary}

👉 Your position:
{'✔ Above market average (competitive)' if salary > avg_salary else '⚠ Below market average'}

### 4. Education Impact
Education contributes to salary but experience has stronger influence in this dataset.
Your qualification (**{edu}**) supports your salary level.

### 5. Skills Impact
Your selected skills:
👉 **{", ".join(skills) if skills else "No skills selected"}**

Skills enhance your value in the job market by improving productivity, specialization, and demand.

## 🧠 Final Insight
This prediction is based on:
✔ Historical dataset patterns
✔ Experience trends
✔ Role-based salary distribution
✔ Skills contribution

👉 The salary ₹{salary} is a **data-driven estimate aligned with market trends**.

"""

    return f"# 💰 Predicted Salary: ₹{salary}", chart, explanation


# ================= STATE =================
def store_outputs(age, gender, edu, job, exp, skills):
    sal, ch, ex = predict(age, gender, edu, job, exp, skills)
    return sal, ch, ex, ch, ex


# ================= APP =================
with gr.Blocks(css=custom_css) as app:

    chart_state = gr.State()
    exp_state = gr.State()

     # 0 HOME
    with gr.Column(visible=True) as p0:
        gr.Markdown("""
# 🚀 AI Salary Prediction System
### Make Smarter Career Decisions with Data
💡 Understand your salary potential using real-world data insights.
### 🔍 Features
✔ Salary prediction
✔ Market comparison
✔ Role-based insights
✔ Clear AI explanation
### 🎯 Who Can Use This?
- Students
- Job Seekers
- Professionals
- HR Analysts


👉 Click below to get started
""")
        start = gr.Button("Get Started")
        about_btn = gr.Button("About")

    # 1 ABOUT
    with gr.Column(visible=False) as p1:
        gr.Markdown("""
## 📘 About

This system uses Machine Learning models to analyze salary data and generate predictions with explanations.
""")
        back1 = gr.Button("Back")


    # 2 LOGIN
    with gr.Column(visible=False) as p2:
        u = gr.Textbox(label="Username")
        pw = gr.Textbox(label="Password", type="password")
        show_pw = gr.Checkbox(label="Show Password")
        show_pw.change(toggle_password, inputs=show_pw, outputs=pw)
        login_btn = gr.Button("Login")
        signup_btn = gr.Button("Signup")
        forgot_btn = gr.Button("Forgot Password")
        msg = gr.Markdown()

    # 3 SIGNUP
    with gr.Column(visible=False) as p3:
        su_user = gr.Textbox(label="Username")
        su_pass = gr.Textbox(label="Password", type="password")
        su_confirm = gr.Textbox(label="Confirm Password", type="password")
        register_btn = gr.Button("Register")
        reg_msg = gr.Markdown()
        back3 = gr.Button("⬅ Back")

    # 4 FORGOT
    with gr.Column(visible=False) as p4:
        fp_user = gr.Textbox()
        fp_btn = gr.Button("Get Password")
        fp_msg = gr.Markdown()
        back4 = gr.Button("⬅ Back")

    # 5 DASHBOARD
    with gr.Column(visible=False) as p5:
        gr.Markdown("## 📊 Dashboard")
        model_info = gr.Button("Model Info")
        go_predict = gr.Button("Predict Salary")
        logout = gr.Button("Logout")
        back5= gr.Button("⬅ Back")

    #  6 MODEL INFO
    with gr.Column(visible=False) as p6:
        gr.Markdown("""
## 📊 Model Information

### 🔍 How this model works
This system uses a **simple rule-based prediction**:
- Base Salary = ₹30,000
- Each year of experience adds ₹7,000

### 📈 Dataset Features
- Years of Experience
- Job Role
- Salary trends

### 🧠 Logic Used
- Experience has strongest impact
- Role average comparison
- Market average benchmarking
""")
        back6 = gr.Button("⬅ Back")


    # 7 INPUT
    with gr.Column(visible=False) as p7:
        age = gr.Number(label="Age", value=25)
        gender = gr.Dropdown(["Male","Female"],label="Gender" , value="Male")
        edu = gr.Dropdown(["Bachelors","Masters","PhD"],label="Education Level", value="Bachelors")
        job = gr.Dropdown(job_roles, value=job_roles[0] if len(job_roles)>0 else None)
        exp = gr.Number(label="Years of Experience", value=0)

        # ✅ SKILLS (WORKING)
        skills = gr.CheckboxGroup(
            choices=skills_list,
            label="Skills",
            value=[]
        )

        predict_btn = gr.Button("Predict")
        back7 = gr.Button("⬅ Back")

        # 8 RESULT
    with gr.Column(visible=False) as p8:
        salary_out = gr.Markdown()
        next_chart = gr.Button("View Chart")
        next_exp = gr.Button("View Explanation")
        back8 = gr.Button("⬅ Back")

    # 9 CHART
    with gr.Column(visible=False) as p9:
        chart = gr.Plot()
        view_exp_btn = gr.Button("View Explanation")
        back9 = gr.Button("⬅ Back")
        home_btn = gr.Button("Home")

    # 10 EXPLANATION
    with gr.Column(visible=False) as p10:
        explanation = gr.Markdown()
        back10 = gr.Button("⬅ Back")
        home10 = gr.Button("Home")

    # 11 PROFILE
    with gr.Column(visible=False) as p11:
        logout2 = gr.Button("Logout")

    # FLOW
    start.click(lambda: show(2), outputs=[p0,p1,p2,p3,p4,p5,p6,p7,p8,p9,p10,p11])
    about_btn.click(lambda: show(1), outputs=[p0,p1,p2,p3,p4,p5,p6,p7,p8,p9,p10,p11])
    back1.click(lambda: show(0), outputs=[p0,p1,p2,p3,p4,p5,p6,p7,p8,p9,p10,p11])

    login_btn.click(login, inputs=[u,pw],
        outputs=[msg,p0,p1,p2,p3,p4,p5,p6,p7,p8,p9,p10,p11])

    signup_btn.click(lambda: show(3), outputs=[p0,p1,p2,p3,p4,p5,p6,p7,p8,p9,p10,p11])
    forgot_btn.click(lambda: show(4), outputs=[p0,p1,p2,p3,p4,p5,p6,p7,p8,p9,p10,p11])

    register_btn.click(signup, inputs=[su_user,su_pass,su_confirm], outputs=reg_msg)
    fp_btn.click(forgot_password, inputs=fp_user, outputs=fp_msg)

    back3.click(lambda: show(2), outputs=[p0,p1,p2,p3,p4,p5,p6,p7,p8,p9,p10,p11])
    back4.click(lambda: show(2), outputs=[p0,p1,p2,p3,p4,p5,p6,p7,p8,p9,p10,p11])

    model_info.click(lambda: show(6), outputs=[p0,p1,p2,p3,p4,p5,p6,p7,p8,p9,p10,p11])
    go_predict.click(lambda: show(7), outputs=[p0,p1,p2,p3,p4,p5,p6,p7,p8,p9,p10,p11])
    logout.click(lambda: show(0), outputs=[p0,p1,p2,p3,p4,p5,p6,p7,p8,p9,p10,p11])

    back6.click(lambda: show(5), outputs=[p0,p1,p2,p3,p4,p5,p6,p7,p8,p9,p10,p11])

    predict_btn.click(
        store_outputs,
        inputs=[age,gender,edu,job,exp, skills],
        outputs=[salary_out, chart, explanation, chart_state, exp_state]
    ).then(lambda: show(8),
        outputs=[p0,p1,p2,p3,p4,p5,p6,p7,p8,p9,p10,p11])

    next_chart.click(lambda ch: ch, inputs=chart_state, outputs=chart)\
        .then(lambda: show(9),
        outputs=[p0,p1,p2,p3,p4,p5,p6,p7,p8,p9,p10,p11])

    next_exp.click(lambda ex: ex, inputs=exp_state, outputs=explanation)\
        .then(lambda: show(10),
        outputs=[p0,p1,p2,p3,p4,p5,p6,p7,p8,p9,p10,p11])

    view_exp_btn.click(lambda ex: ex, inputs=exp_state, outputs=explanation)\
        .then(lambda: show(10),
        outputs=[p0,p1,p2,p3,p4,p5,p6,p7,p8,p9,p10,p11])

    back7.click(lambda: show(5), outputs=[p0,p1,p2,p3,p4,p5,p6,p7,p8,p9,p10,p11])
    back8.click(lambda: show(7), outputs=[p0,p1,p2,p3,p4,p5,p6,p7,p8,p9,p10,p11])
    back9.click(lambda: show(8), outputs=[p0,p1,p2,p3,p4,p5,p6,p7,p8,p9,p10,p11])
    back10.click(lambda: show(8), outputs=[p0,p1,p2,p3,p4,p5,p6,p7,p8,p9,p10,p11])

    home_btn.click(lambda: show(0), outputs=[p0,p1,p2,p3,p4,p5,p6,p7,p8,p9,p10,p11])
    home10.click(lambda: show(0), outputs=[p0,p1,p2,p3,p4,p5,p6,p7,p8,p9,p10,p11])

    logout2.click(lambda: show(0), outputs=[p0,p1,p2,p3,p4,p5,p6,p7,p8,p9,p10,p11])

app.launch()


/tmp/ipykernel_6653/2739673435.py:197: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css) as app:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://51605736cdc9435ba1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
